dataset.py

In [3]:
import sys; sys.path.insert(0, "src")
import eda, splits, dataset

wide = eda.group_rare_species(eda.to_wide(eda.load_long()))
folds = splits.get_cv_folds(wide, n_splits=5)
tr, vl = dataset.make_fold_loaders(wide, folds[0], batch_size=8, num_workers=0)

imgs, ys = next(iter(tr))
print("batch images:", imgs.shape)   # ожидается [8, 3, 224, 224]
print("batch targets:", ys.shape)    # [8, 5]

c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


batch images: torch.Size([8, 3, 224, 224])
batch targets: torch.Size([8, 5])


model.py

In [1]:
import sys; sys.path.insert(0, "src")
import torch, model

for kind in ["simple_cnn", "resnet18", "efficientnet_b0"]:
    m = model.build_model(kind, pretrained=False)  # pretrained=False для быстрой проверки
    out = m(torch.randn(2, 3, 224, 224))
    print(f"{kind:18} out={tuple(out.shape)}  trainable={model.count_trainable(m):,}")

simple_cnn         out=(2, 5)  trainable=422,917
resnet18           out=(2, 5)  trainable=11,179,077
efficientnet_b0    out=(2, 5)  trainable=4,013,953


In [4]:
m_frozen = model.build_model("resnet18", pretrained=False, freeze_backbone=True)
print("resnet18 frozen trainable:", model.count_trainable(m_frozen))  # ожидается ~2,565 (только голова)

resnet18 frozen trainable: 2565


train.py
fast smoke-test for 1 fold, small epoch

In [ ]:
import sys; sys.path.insert(0, "src")
import eda, train

wide = eda.group_rare_species(eda.to_wide(eda.load_long()))

# smoke: resnet18, 2 фолда, мало эпох, num_workers=0 если Windows
res = train.cross_validate(
    wide, kind="resnet18", n_splits=5,
    batch_size=32, num_workers=2,
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    train_kwargs={"max_epochs": 15, "patience": 5},
    verbose=True,
)
print("weighted RMSE:", res["weighted_rmse_mean"], "±", res["weighted_rmse_std"])
print(res["summary"])

[fold 0] resnet18 on cpu


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 01 | train_loss 3.3871 | val_wRMSE 58.210


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 02 | train_loss 1.1236 | val_wRMSE 1100052249.717


...

  epoch 03 | train_loss 0.8340 | val_wRMSE 609.501


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 04 | train_loss 0.7136 | val_wRMSE 24.667


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 05 | train_loss 0.6684 | val_wRMSE 40.557
weighted RMSE: 27.460767059326173 ± 15.824939900693469
                         rmse             mae              r2        
                         mean     std    mean     std    mean     std
model    target                                                      
resnet18 Dry_Clover_g  11.935   2.391   5.559   1.217   0.001   0.279
         Dry_Dead_g    37.428  56.205  22.944  33.864 -22.093  49.177
         Dry_Green_g   24.162  10.556  16.363   7.788  -0.167   0.912
         Dry_Total_g   29.874  17.924  22.291  13.965  -0.430   1.576
         GDM_g         25.857  12.258  18.143   9.377  -0.434   1.184


In [ ]:
import sys; sys.path.insert(0, "src")
import eda, train

wide = eda.group_rare_species(eda.to_wide(eda.load_long()))

r_frozen = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    n_splits=5, batch_size=32, num_workers=0,
    train_kwargs={"max_epochs": 50, "patience": 8, "lr": 1e-3},
    verbose=True,
)
print("frozen weighted RMSE:", r_frozen["weighted_rmse_mean"], "±", r_frozen["weighted_rmse_std"])
print(r_frozen["summary"])

In [7]:
import sys; sys.path.insert(0, "src")
import eda, train

wide = eda.group_rare_species(eda.to_wide(eda.load_long()))

common = dict(
    n_splits=5,
    img_size=224,
    batch_size=32,
    num_workers=0,        # 0 на Windows локально; 2 в Colab
    train_kwargs={"max_epochs": 50, "patience": 8, "lr": 1e-3},
    verbose=True,
)

# 1. CNN с нуля (session 10)
r_cnn = train.cross_validate(
    wide, kind="simple_cnn",
    model_kwargs={},
    **common,
)

# 2. ResNet18 feature-extraction (backbone заморожен)
r_frozen = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": True},
    **common,
)

# 3. ResNet18 fine-tuning (backbone разморожен)
r_ft = train.cross_validate(
    wide, kind="resnet18",
    model_kwargs={"pretrained": True, "freeze_backbone": False},
    **common,
)

[fold 0] simple_cnn on cpu


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 01 | train_loss 4.1982 | val_wRMSE 45.851


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 02 | train_loss 1.3982 | val_wRMSE 35.138


...

  epoch 14 | train_loss 0.8652 | val_wRMSE 14.879
  early stop at epoch 14 (best wRMSE 14.143)
[fold 2] simple_cnn on cpu


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  epoch 01 | train_loss 4.2995 | val_wRMSE 42.837


c:\Python310\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


KeyboardInterrupt: 